# Chapter 1: The Governance Problem for Agentic AI

**Learning objectives:**
1. Why unconstrained agentic AI fails (trajectory failures, guardrail illusion, cognitive mismatch)
2. The five design principles: policy primacy, computable governance, separation of reasoning and authority, human-over-the-loop, formal contracts
3. The six-layer architecture overview
4. Preview what the governed agent will look like

In [ ]:
# === Preamble (all notebooks) ===

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}, CUDA: {torch.cuda.is_available()}")

import pandas as pd
from sklearn.model_selection import train_test_split

## Key Terminology

Three terms recur throughout this monograph and deserve precise definitions before we begin.

**Agentic AI system.** A software system in which a large language model (LLM) autonomously selects and executes a sequence of *actions* --- tool calls, API requests, code execution --- to accomplish a goal. The critical difference from a traditional ML model is that the system produces an **execution trajectory**: an ordered sequence of states, actions and observations $\tau = (s_0, a_0, o_0, s_1, a_1, o_1, \ldots, s_T)$ rather than a single input-output mapping $f(x) \to y$. Governance failures in agentic systems are therefore *trajectory-level* failures: skipped steps, unauthorized tool calls, missing approvals or emergent loops.

**Governance.** The set of policies, processes and enforcement mechanisms that ensure an AI system operates within organizational, legal and ethical boundaries. Governance goes beyond model performance: it encompasses fairness, robustness, auditability, provenance and human oversight.

**Guardrail vs. contract.** A **guardrail** is a text instruction embedded in a prompt that *asks* an agent to follow a rule (e.g., "always check fairness before deployment"). Nothing enforces it. A **contract** is a structural mechanism --- such as a finite state machine --- that *physically blocks* rule violations. The agent cannot bypass a contract because the enforcement is in code, not in language. This distinction is the central theme of this monograph.

## Lab 1.1 --- See the Governance Problem Firsthand

### Part A: Iris --- High Accuracy, Zero Governance

We train a quick-and-dirty classifier on Iris. The model *works* --- accuracy is high --- but the pipeline has **no governance**: no audit trail, no preprocessing documentation, no fairness check, no robustness evaluation, no model comparison, no provenance.

In [2]:
# --- Lab 1.1a: Ungoverned Iris pipeline ---
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Iris accuracy: {acc:.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=iris.target_names)}")

Iris accuracy: 1.0000

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        19
  versicolor       1.00      1.00      1.00        13
   virginica       1.00      1.00      1.00        13

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45



In [3]:
# --- Governance checklist for the Iris pipeline ---
governance_checklist = {
    "Data provenance recorded":           False,
    "Preprocessing steps documented":     False,
    "Alternative models compared":        False,
    "Robustness to noise evaluated":      False,
    "Fairness across subgroups checked":  False,
    "Audit trail / decision log exists":  False,
    "Model card or documentation":        False,
    "Human sign-off before deployment":   False,
}

print("Governance checklist for the Iris pipeline:")
print("-" * 50)
for item, status in governance_checklist.items():
    mark = "PASS" if status else "FAIL"
    print(f"  [{mark}] {item}")
print(f"\nGovernance score: 0 / {len(governance_checklist)}")
print(f"\n>>> The model is accurate ({acc:.1%}) but completely ungoverned.")

Governance checklist for the Iris pipeline:
--------------------------------------------------
  [FAIL] Data provenance recorded
  [FAIL] Preprocessing steps documented
  [FAIL] Alternative models compared
  [FAIL] Robustness to noise evaluated
  [FAIL] Fairness across subgroups checked
  [FAIL] Audit trail / decision log exists
  [FAIL] Model card or documentation
  [FAIL] Human sign-off before deployment

Governance score: 0 / 8

>>> The model is accurate (100.0%) but completely ungoverned.


### Part B: Adult Income --- Good Accuracy, Hidden Fairness Issues

Now we scale up to a dataset where governance matters even more. The Adult Income dataset predicts whether a person earns more than $50K/year. It contains sensitive attributes (sex, race) that create real fairness risks. An ungoverned pipeline will miss them entirely.

In [4]:
# --- Lab 1.1b: Ungoverned Adult Income pipeline ---
from sklearn.datasets import fetch_openml
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder

# Load dataset
adult = fetch_openml("adult", version=2, as_frame=True, parser="auto")
df = adult.data.copy()
df["income"] = (adult.target == ">50K").astype(int)

print(f"Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Target distribution:\n{df['income'].value_counts().to_string()}")
print("\nSensitive columns present: sex, race")
print(f"  sex values:  {df['sex'].unique().tolist()}")
print(f"  race values: {df['race'].unique().tolist()}")

Dataset: 48,842 rows, 15 columns
Target distribution:
income
0    37155
1    11687

Sensitive columns present: sex, race
  sex values:  ['Male', 'Female']
  race values: ['Black', 'White', 'Asian-Pac-Islander', 'Other', 'Amer-Indian-Eskimo']


In [5]:
# --- Train a quick model (ungoverned) ---
# Simple preprocessing: encode categoricals, drop missing
df_clean = df.dropna()

# Separate features and target
y = df_clean["income"]
X = df_clean.drop(columns=["income"])

# Encode categoricals
le_dict = {}
for col in X.select_dtypes(include=["category", "object"]).columns:
    le_dict[col] = LabelEncoder()
    X[col] = le_dict[col].fit_transform(X[col].astype(str))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Train a GBM --- the "data scientist who just wants accuracy"
gbm = GradientBoostingClassifier(
    n_estimators=100, max_depth=3, random_state=42
)
gbm.fit(X_train, y_train)
y_pred = gbm.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Adult Income accuracy: {acc:.4f}")
print("\n>>> A data scientist might stop here. 'Accuracy is good, ship it.'")

Adult Income accuracy: 0.8656

>>> A data scientist might stop here. 'Accuracy is good, ship it.'


### Fairness Metrics: Selection Rate and Adverse Impact Ratio

To assess whether the model treats different groups equitably, we compute two metrics:

- **Selection rate**: the fraction of individuals in a subgroup whom the model predicts as positive (income >\$50K).
- **Adverse Impact Ratio (AIR)**: the ratio of the lowest selection rate to the highest across subgroups. Under the *four-fifths rule* (EEOC Uniform Guidelines, 29 CFR §1607.4D) used in US employment law, an AIR below 0.80 signals potential **disparate impact** --- a pattern in which a facially neutral model disproportionately harms a protected group.

We will formalize these fairness requirements as machine-checkable policy rules in Chapter 12 and enforce them through release gates in Chapter 15.

In [6]:
# --- The governance problem: hidden fairness issues ---
# Compute accuracy and selection rates by sex
test_df = X_test.copy()
test_df["y_true"] = y_test.values
test_df["y_pred"] = y_pred

# Recover original sex labels
sex_col = "sex"
sex_map = dict(enumerate(le_dict[sex_col].classes_))
test_df["sex_label"] = test_df[sex_col].map(sex_map)

print("=" * 60)
print("FAIRNESS ANALYSIS (what the ungoverned pipeline missed)")
print("=" * 60)

for group in test_df["sex_label"].unique():
    mask = test_df["sex_label"] == group
    group_acc = accuracy_score(test_df.loc[mask, "y_true"],
                               test_df.loc[mask, "y_pred"])
    # Selection rate: fraction predicted as high-income
    sel_rate = test_df.loc[mask, "y_pred"].mean()
    n = mask.sum()
    print(f"\n  {group}:")
    print(f"    N = {n:,}")
    print(f"    Accuracy      = {group_acc:.4f}")
    print(f"    Selection rate = {sel_rate:.4f}  (predicted >50K)")

# Compute adverse impact ratio
sel_rates = test_df.groupby("sex_label")["y_pred"].mean()
if len(sel_rates) == 2:
    air = sel_rates.min() / sel_rates.max()
    advantaged = sel_rates.idxmax()
    disadvantaged = sel_rates.idxmin()
    print(f"\n  Adverse Impact Ratio (AIR) = {air:.4f}")
    print(f"    ({disadvantaged} / {advantaged})")
    if air < 0.8:
        print("    >>> BELOW 0.80 threshold -- potential disparate impact!")
    else:
        print("    >>> Above 0.80 threshold")

print(f"\n>>> An ungoverned pipeline reported {accuracy_score(y_test, y_pred):.1%} accuracy")
print(">>> but never checked for disparate impact across protected groups.")

FAIRNESS ANALYSIS (what the ungoverned pipeline missed)

  Female:
    N = 4,334
    Accuracy      = 0.9352
    Selection rate = 0.0718  (predicted >50K)

  Male:
    N = 9,233
    Accuracy      = 0.8329
    Selection rate = 0.2403  (predicted >50K)

  Adverse Impact Ratio (AIR) = 0.2986
    (Female / Male)
    >>> BELOW 0.80 threshold -- potential disparate impact!

>>> An ungoverned pipeline reported 86.6% accuracy
>>> but never checked for disparate impact across protected groups.


In [7]:
# --- Fairness by race ---
race_col = "race"
race_map = dict(enumerate(le_dict[race_col].classes_))
test_df["race_label"] = test_df[race_col].map(race_map)

print("Selection rates by race:")
print("-" * 45)
race_sel = test_df.groupby("race_label")["y_pred"].mean().sort_values(ascending=False)
for race, rate in race_sel.items():
    n = (test_df["race_label"] == race).sum()
    print(f"  {race:<25s}  {rate:.4f}  (N={n:,})")

max_rate = race_sel.max()
print(f"\nAdverse Impact Ratios (vs. {race_sel.idxmax()}):")
for race, rate in race_sel.items():
    air_r = rate / max_rate
    flag = " <<<" if air_r < 0.8 else ""
    print(f"  {race:<25s}  AIR = {air_r:.4f}{flag}")

Selection rates by race:
---------------------------------------------
  Asian-Pac-Islander         0.2632  (N=380)
  White                      0.1977  (N=11,689)
  Black                      0.0814  (N=1,253)
  Other                      0.0762  (N=105)
  Amer-Indian-Eskimo         0.0643  (N=140)

Adverse Impact Ratios (vs. Asian-Pac-Islander):
  Asian-Pac-Islander         AIR = 1.0000
  White                      AIR = 0.7513 <<<
  Black                      AIR = 0.3093 <<<
  Other                      AIR = 0.2895 <<<
  Amer-Indian-Eskimo         AIR = 0.2443 <<<


## Lab 1.2 --- Why Prompt Guardrails Fail

A common response to governance concerns is to add text-based guardrails: "Tell the agent not to do X in the system prompt." This lab demonstrates why that approach is fundamentally insufficient.

We compare two approaches:
1. **Text policy (guardrail):** A string that *asks* the agent to follow a rule. Nothing enforces it.
2. **Structural constraint (contract):** A state machine that *physically blocks* rule violations.

In [8]:
# --- Lab 1.2a: Text policy (guardrail) ---
# This is how most teams "govern" their agents today.

TEXT_POLICY = """
POLICY: You must follow this tool-calling order:
  1. Always call 'load_data' before 'train_model'.
  2. Always call 'check_fairness' before 'deploy'.
  3. Never call 'deploy' without human approval.
"""

# Simulate an agent that receives this policy as a prompt prefix.
# The agent's "actions" are just function calls --- nothing enforces the order.

def load_data():
    return {"status": "loaded", "rows": 48842}

def train_model():
    return {"status": "trained", "accuracy": 0.864}

def check_fairness():
    return {"status": "checked", "air": 0.38, "pass": False}

def deploy():
    return {"status": "DEPLOYED TO PRODUCTION"}

# A reckless agent ignores the policy and calls tools in the wrong order
print("Text policy says: load_data -> train_model -> check_fairness -> deploy")
print("=" * 60)
print("\nReckless agent execution (ignoring policy):")

actions = [
    ("train_model",     train_model),      # Skipped load_data!
    ("deploy",          deploy),            # Skipped check_fairness!
]

trace = []
for name, fn in actions:
    result = fn()
    trace.append({"tool": name, "result": result})
    print(f"  Step: {name:20s} -> {result}")

print("\n>>> The text policy was IGNORED. The model was deployed without")
print(">>> loading data, checking fairness, or getting human approval.")
print(">>> Nothing in the system prevented this.")

Text policy says: load_data -> train_model -> check_fairness -> deploy

Reckless agent execution (ignoring policy):
  Step: train_model          -> {'status': 'trained', 'accuracy': 0.864}
  Step: deploy               -> {'status': 'DEPLOYED TO PRODUCTION'}

>>> The text policy was IGNORED. The model was deployed without
>>> loading data, checking fairness, or getting human approval.
>>> Nothing in the system prevented this.


### What Is a Finite State Machine (FSM)?

A **finite state machine** is a model of computation defined by a finite set of *states*, a set of allowed *actions* (inputs) and a *transition function* that maps each (state, action) pair to a next state. If no transition exists for a given pair, the action is blocked. FSMs are deterministic, auditable and immune to prompt injection because the enforcement mechanism is a lookup table, not a language model. We develop FSM-based behavioral contracts fully in Chapter 13; here we build a minimal example.

In [9]:
# --- Lab 1.2b: Structural constraint (FSM behavioral contract) ---
# Instead of asking the agent to follow the policy, we ENFORCE it.

class WorkflowFSM:
    """A finite state machine that physically blocks policy violations.

    The agent proposes actions; the FSM decides whether each action
    is allowed given the current state.  This is a 'contract' ---
    not a suggestion.
    """
    def __init__(self):
        self.state = "start"
        self.transitions = {
            "start":          {"load_data":       "data_loaded"},
            "data_loaded":    {"train_model":     "model_trained"},
            "model_trained":  {"check_fairness":  "fairness_checked"},
            "fairness_checked": {"deploy":        "deployed"},
            # deploy requires fairness_checked --- no shortcut exists
        }
        self.log = []

    def attempt(self, action: str) -> dict:
        """Attempt a state transition. Returns result dict."""
        allowed = self.transitions.get(self.state, {})
        if action in allowed:
            old_state = self.state
            self.state = allowed[action]
            entry = {
                "action": action, "allowed": True,
                "from": old_state, "to": self.state,
            }
            self.log.append(entry)
            return entry
        else:
            entry = {
                "action": action, "allowed": False,
                "from": self.state, "to": self.state,
                "reason": f"'{action}' not allowed in state '{self.state}'. "
                          f"Allowed: {list(allowed.keys())}",
            }
            self.log.append(entry)
            return entry


# Same reckless agent, but now governed by the FSM
fsm = WorkflowFSM()

print("Structural contract (FSM) enforcement:")
print("=" * 60)

reckless_actions = ["train_model", "deploy"]  # Same bad sequence
for action in reckless_actions:
    result = fsm.attempt(action)
    status = "ALLOWED" if result["allowed"] else "BLOCKED"
    print(f"\n  Agent tries: {action}")
    print(f"  FSM state:   {result['from']}")
    print(f"  Result:      {status}")
    if not result["allowed"]:
        print(f"  Reason:      {result['reason']}")

print("\n>>> The FSM BLOCKED both illegal actions.")
print(">>> The agent cannot skip steps or deploy without fairness checks.")

Structural contract (FSM) enforcement:

  Agent tries: train_model
  FSM state:   start
  Result:      BLOCKED
  Reason:      'train_model' not allowed in state 'start'. Allowed: ['load_data']

  Agent tries: deploy
  FSM state:   start
  Result:      BLOCKED
  Reason:      'deploy' not allowed in state 'start'. Allowed: ['load_data']

>>> The FSM BLOCKED both illegal actions.
>>> The agent cannot skip steps or deploy without fairness checks.


In [10]:
# --- Lab 1.2c: The correct workflow under FSM governance ---
fsm2 = WorkflowFSM()

print("Correct workflow under FSM governance:")
print("=" * 60)

correct_actions = ["load_data", "train_model", "check_fairness", "deploy"]
for action in correct_actions:
    result = fsm2.attempt(action)
    status = "ALLOWED" if result["allowed"] else "BLOCKED"
    print(f"  {action:20s}  {result['from']:20s} -> {result['to']:20s}  [{status}]")

print(f"\nFSM audit log ({len(fsm2.log)} entries):")
for i, entry in enumerate(fsm2.log, 1):
    print(f"  [{i}] {entry['action']:20s}  {entry['from']} -> {entry['to']}  "
          f"allowed={entry['allowed']}")

print("\n>>> Every action is logged. Every transition is verifiable.")
print(">>> This is the difference between a guardrail and a contract.")

Correct workflow under FSM governance:
  load_data             start                -> data_loaded           [ALLOWED]
  train_model           data_loaded          -> model_trained         [ALLOWED]
  check_fairness        model_trained        -> fairness_checked      [ALLOWED]
  deploy                fairness_checked     -> deployed              [ALLOWED]

FSM audit log (4 entries):
  [1] load_data             start -> data_loaded  allowed=True
  [2] train_model           data_loaded -> model_trained  allowed=True
  [3] check_fairness        model_trained -> fairness_checked  allowed=True
  [4] deploy                fairness_checked -> deployed  allowed=True

>>> Every action is logged. Every transition is verifiable.
>>> This is the difference between a guardrail and a contract.


## The Five Design Principles

The failures we observed in Labs 1.1 and 1.2 motivate five principles for building policy-governed AI agents:

| # | Principle | What It Means | Lab Evidence |
|---|-----------|---------------|--------------|
| 1 | **Policy Primacy** | Organizational policy is the *source of truth*, not the model's learned behavior. | Lab 1.1: the model optimized accuracy, not fairness --- policy was absent. |
| 2 | **Computable Governance** | Every policy must be machine-checkable, not just human-readable prose. | Lab 1.2a: the text policy was unenforceable. |
| 3 | **Separation of Reasoning and Authority** | The agent *proposes* actions; a separate mechanism *authorizes* them. | Lab 1.2b: the FSM (authority) is separate from the agent (reasoning). |
| 4 | **Human-over-the-Loop** | Humans set policy and review escalations but do not approve every action. | Lab 1.1b: no human ever reviewed the fairness implications. |
| 5 | **Formal Contracts** | Agent behavior is bounded by verifiable state-machine contracts, not prompt suggestions. | Lab 1.2b vs 1.2a: contract blocks violations; text does not. |

## The Six-Layer Architecture

These principles lead to a six-layer architecture for policy-governed agentic AI. Each layer has a distinct responsibility, and no layer trusts the layer above it.

In [11]:
# --- The six-layer architecture ---
layers = [
    ("Layer 1", "Knowledge Foundation",
     "GMS ingests domain knowledge into a geometric knowledge graph on S^{m-1}. "
     "This is the ground truth the agent reasons over."),
    ("Layer 2", "Tool Gateway",
     "Every tool call passes through a gateway that enforces pre/post-conditions, "
     "rate limits and data-type contracts."),
    ("Layer 3", "Behavioral Contracts",
     "FSM-based workflow specifications define which tool sequences are legal. "
     "The agent cannot deviate from the contracted workflow."),
    ("Layer 4", "Policy Engine",
     "Organizational rules (fairness thresholds, approval requirements, "
     "data-handling restrictions) are compiled into machine-checkable predicates."),
    ("Layer 5", "Verification & Telemetry",
     "Every claim the agent makes is verified against GMS facts. Every action "
     "is logged to an immutable audit trail."),
    ("Layer 6", "Orchestration & Release Gates",
     "A governed orchestrator sequences the full pipeline and enforces "
     "release gates before any model reaches production."),
]

print("Six-Layer Architecture for Policy-Governed Agentic AI")
print("=" * 60)
for num, name, desc in layers:
    print(f"\n  {num}: {name}")
    print(f"    {desc}")

print("\n" + "=" * 60)
print("Each layer will be built in a dedicated chapter of this monograph.")

Six-Layer Architecture for Policy-Governed Agentic AI

  Layer 1: Knowledge Foundation
    GMS ingests domain knowledge into a geometric knowledge graph on S^{m-1}. This is the ground truth the agent reasons over.

  Layer 2: Tool Gateway
    Every tool call passes through a gateway that enforces pre/post-conditions, rate limits and data-type contracts.

  Layer 3: Behavioral Contracts
    FSM-based workflow specifications define which tool sequences are legal. The agent cannot deviate from the contracted workflow.

  Layer 4: Policy Engine
    Organizational rules (fairness thresholds, approval requirements, data-handling restrictions) are compiled into machine-checkable predicates.

  Layer 5: Verification & Telemetry
    Every claim the agent makes is verified against GMS facts. Every action is logged to an immutable audit trail.

  Layer 6: Orchestration & Release Gates
    A governed orchestrator sequences the full pipeline and enforces release gates before any model reaches produ

## Preview: What the Governed Agent Will Look Like

By the end of this monograph, we will have built a complete governed data-science agent. Here is a preview of the contrast between the ungoverned pipeline from Lab 1.1 and the governed pipeline we will build.

In [12]:
# --- Preview: ungoverned vs governed pipeline ---
comparison = {
    "Step":                  ["Load data",         "Preprocess",        "Train model",
                              "Evaluate accuracy",  "Check fairness",    "Check robustness",
                              "Compare models",     "Document",          "Release gate",
                              "Deploy"],
    "Ungoverned (Lab 1.1)":  ["Manual",            "Ad hoc",            "Single model",
                              "Yes (only metric)",  "SKIPPED",           "SKIPPED",
                              "SKIPPED",            "SKIPPED",           "SKIPPED",
                              "Manual push"],
    "Governed (This Book)":  ["Tool + provenance", "Contracted pipeline","GMS-guided selection",
                              "Multi-metric suite", "AIR + subgroup",    "Noise perturbation",
                              "Leaderboard + GMS",  "Auto model card",   "Policy gate (FSM)",
                              "Human-approved"],
}

comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))

# Count governance gaps
gaps = sum(1 for v in comparison["Ungoverned (Lab 1.1)"] if "SKIP" in v)
print(f"\n>>> The ungoverned pipeline skips {gaps} of 10 governance steps.")
print(">>> The governed pipeline enforces all 10 through structural contracts.")

             Step Ungoverned (Lab 1.1) Governed (This Book)
        Load data               Manual    Tool + provenance
       Preprocess               Ad hoc  Contracted pipeline
      Train model         Single model GMS-guided selection
Evaluate accuracy    Yes (only metric)   Multi-metric suite
   Check fairness              SKIPPED       AIR + subgroup
 Check robustness              SKIPPED   Noise perturbation
   Compare models              SKIPPED    Leaderboard + GMS
         Document              SKIPPED      Auto model card
     Release gate              SKIPPED    Policy gate (FSM)
           Deploy          Manual push       Human-approved

>>> The ungoverned pipeline skips 5 of 10 governance steps.
>>> The governed pipeline enforces all 10 through structural contracts.


## Exercises

### Exercise 1.1 --- Real-World Governance Failures
List three real-world AI governance failures (from news, regulation or your own experience) and map each to one of the five design principles that would have prevented or mitigated it.

In [13]:
# --- Exercise 1.1: Map governance failures to design principles ---
# Example solution (students should provide their own)

failures = [
    {
        "incident": "Amazon hiring tool discriminated against women (2018)",
        "description": "ML model trained on historical hiring data learned to "
                       "penalize resumes containing the word 'women's'.",
        "principle": "Policy Primacy",
        "explanation": "Organizational fairness policy should have been the "
                       "source of truth, overriding biased training data. "
                       "The model optimized for historical patterns, not policy.",
    },
    {
        "incident": "Tesla Autopilot phantom braking (2021-2023)",
        "description": "Autonomous driving system applied emergency braking "
                       "based on false obstacle detection with no human override.",
        "principle": "Human-over-the-Loop",
        "explanation": "Safety-critical decisions (emergency braking) should "
                       "have human oversight. The system acted autonomously "
                       "in a domain where human review was essential.",
    },
    {
        "incident": "ChatGPT data leak via prompt injection (2023)",
        "description": "Adversarial prompts caused the model to reveal system "
                       "prompts and prior conversation data.",
        "principle": "Formal Contracts",
        "explanation": "Text-based instructions ('do not reveal your system prompt') "
                       "are guardrails, not contracts. A structural constraint "
                       "(output filter, state machine) would have physically "
                       "blocked the information leak.",
    },
]

print("Exercise 1.1: Real-World Governance Failures")
print("=" * 60)
for i, f in enumerate(failures, 1):
    print(f"\n{i}. {f['incident']}")
    print(f"   Description: {f['description']}")
    print(f"   Principle:   {f['principle']}")
    print(f"   Explanation: {f['explanation']}")

Exercise 1.1: Real-World Governance Failures

1. Amazon hiring tool discriminated against women (2018)
   Description: ML model trained on historical hiring data learned to penalize resumes containing the word 'women's'.
   Principle:   Policy Primacy
   Explanation: Organizational fairness policy should have been the source of truth, overriding biased training data. The model optimized for historical patterns, not policy.

2. Tesla Autopilot phantom braking (2021-2023)
   Description: Autonomous driving system applied emergency braking based on false obstacle detection with no human override.
   Principle:   Human-over-the-Loop
   Explanation: Safety-critical decisions (emergency braking) should have human oversight. The system acted autonomously in a domain where human review was essential.

3. ChatGPT data leak via prompt injection (2023)
   Description: Adversarial prompts caused the model to reveal system prompts and prior conversation data.
   Principle:   Formal Contracts
   Exp

### Exercise 1.2 --- Classify Governance Requirements into Architecture Layers
Given ten governance requirements, classify each into the appropriate layer of the six-layer architecture.

In [14]:
# --- Exercise 1.2: Classify requirements into architecture layers ---

requirements = [
    ("The agent must not deploy a model with AIR < 0.80.",              "Layer 4: Policy Engine"),
    ("All tool calls must be logged with timestamps.",                   "Layer 5: Verification & Telemetry"),
    ("Domain knowledge about credit risk must be ingested first.",       "Layer 1: Knowledge Foundation"),
    ("The agent must call check_fairness before deploy.",                "Layer 3: Behavioral Contracts"),
    ("Tool inputs must match the declared JSON schema.",                 "Layer 2: Tool Gateway"),
    ("A human must approve any model for production release.",           "Layer 6: Orchestration & Release Gates"),
    ("The agent's accuracy claims must be verified against test data.",  "Layer 5: Verification & Telemetry"),
    ("Rate-limit the agent to at most 10 tool calls per minute.",       "Layer 2: Tool Gateway"),
    ("The training workflow must follow: data -> train -> evaluate.",    "Layer 3: Behavioral Contracts"),
    ("All models must pass a robustness threshold before release.",      "Layer 6: Orchestration & Release Gates"),
]

layer_names = {
    "Layer 1": "Knowledge Foundation",
    "Layer 2": "Tool Gateway",
    "Layer 3": "Behavioral Contracts",
    "Layer 4": "Policy Engine",
    "Layer 5": "Verification & Telemetry",
    "Layer 6": "Orchestration & Release Gates",
}

print("Exercise 1.2: Classify Requirements into Architecture Layers")
print("=" * 70)
for i, (req, layer) in enumerate(requirements, 1):
    print(f"\n  {i:2d}. \"{req}\"")
    print(f"      -> {layer}")

Exercise 1.2: Classify Requirements into Architecture Layers

   1. "The agent must not deploy a model with AIR < 0.80."
      -> Layer 4: Policy Engine

   2. "All tool calls must be logged with timestamps."
      -> Layer 5: Verification & Telemetry

   3. "Domain knowledge about credit risk must be ingested first."
      -> Layer 1: Knowledge Foundation

   4. "The agent must call check_fairness before deploy."
      -> Layer 3: Behavioral Contracts

   5. "Tool inputs must match the declared JSON schema."
      -> Layer 2: Tool Gateway

   6. "A human must approve any model for production release."
      -> Layer 6: Orchestration & Release Gates

   7. "The agent's accuracy claims must be verified against test data."
      -> Layer 5: Verification & Telemetry

   8. "Rate-limit the agent to at most 10 tool calls per minute."
      -> Layer 2: Tool Gateway

   9. "The training workflow must follow: data -> train -> evaluate."
      -> Layer 3: Behavioral Contracts

  10. "All models

### Exercise 1.3 --- Guardrail vs Contract
Explain the difference between "try not to do X" (guardrail) and "cannot do X" (contract). Demonstrate with a concrete code example extending the FSM from Lab 1.2.

In [15]:
# --- Exercise 1.3: Guardrail vs Contract ---

# GUARDRAIL approach: a text instruction
guardrail = "RULE: Never deploy a model with accuracy below 0.80."

# The guardrail is just a string. Nothing prevents violation.
model_accuracy = 0.72  # Below threshold
print("GUARDRAIL approach:")
print(f"  Policy text: '{guardrail}'")
print(f"  Model accuracy: {model_accuracy}")
print("  Agent deploys anyway: deploy() -> DEPLOYED")
print("  Violation detected? NO (nothing checked)\n")

# CONTRACT approach: a function that physically blocks the action
class DeploymentGate:
    """A contract that blocks deployment if accuracy < threshold."""
    def __init__(self, min_accuracy: float = 0.80):
        self.min_accuracy = min_accuracy
        self.log = []

    def check(self, accuracy: float) -> dict:
        allowed = accuracy >= self.min_accuracy
        result = {
            "allowed": allowed,
            "accuracy": accuracy,
            "threshold": self.min_accuracy,
            "reason": (f"accuracy {accuracy:.4f} >= {self.min_accuracy}"
                      if allowed else
                      f"accuracy {accuracy:.4f} < {self.min_accuracy} -- BLOCKED"),
        }
        self.log.append(result)
        return result

gate = DeploymentGate(min_accuracy=0.80)

print("CONTRACT approach:")
# Attempt with low accuracy
result1 = gate.check(0.72)
print(f"  Attempt 1: accuracy=0.72 -> {'ALLOWED' if result1['allowed'] else 'BLOCKED'}")
print(f"    Reason: {result1['reason']}")

# Attempt with sufficient accuracy
result2 = gate.check(0.864)
print(f"  Attempt 2: accuracy=0.864 -> {'ALLOWED' if result2['allowed'] else 'BLOCKED'}")
print(f"    Reason: {result2['reason']}")

print(f"\n  Audit log: {len(gate.log)} entries recorded")
print("\n>>> GUARDRAIL = 'try not to do X' (advisory, unenforceable)")
print(">>> CONTRACT  = 'cannot do X' (structural, machine-enforced)")

GUARDRAIL approach:
  Policy text: 'RULE: Never deploy a model with accuracy below 0.80.'
  Model accuracy: 0.72
  Agent deploys anyway: deploy() -> DEPLOYED
  Violation detected? NO (nothing checked)

CONTRACT approach:
  Attempt 1: accuracy=0.72 -> BLOCKED
    Reason: accuracy 0.7200 < 0.8 -- BLOCKED
  Attempt 2: accuracy=0.864 -> ALLOWED
    Reason: accuracy 0.8640 >= 0.8

  Audit log: 2 entries recorded

>>> GUARDRAIL = 'try not to do X' (advisory, unenforceable)
>>> CONTRACT  = 'cannot do X' (structural, machine-enforced)
